In this notebook, we will extract information from a list of YouTube videos using pandas, the YouTube API, and the pytube library. The resulting data will be saved as a Delta table in S3.

## Initial Phase: Imports and Authentication

In [ ]:
# Installing required libraries: pytube, pandas, deltalake, and boto3

!pip install pytube
!pip install pandas
!pip install pytest
!pip install boto3

In [ ]:
# Imports

from googleapiclient.discovery import build

import pandas as pd

from IPython.display import JSON

import os

from pytube import YouTube, Search, Channel, Playlist, extract

from getpass import getpass

import boto3

## Authentication

In [ ]:
# Basic configuration
api_key = 'YOUR_API_KEY_HERE'
api_service_name = "youtube"
api_version = "v3"

# Authenticating with the YouTube API
youtube_client_obj = build(api_service_name, api_version, developerKey=api_key)

## Execution Phase: Class and Methods

### Building the Project Data

In [ ]:
from typing import List
import pandas as pd


class YouTubeVideoDetails:
    """Class for retrieving YouTube video details via the YouTube Data API.

    The constructor initializes an instance by receiving the authenticated
    YouTube client object as a parameter.
    """

    def __init__(self, youtube_client):
        self.youtube = youtube_client

    def _fetch_video_details(self, video_ids: List[str]) -> List[dict]:
        """Fetches details for a list of YouTube video IDs.

        Uses type annotations to indicate that video_ids is a list of strings,
        and returns a list of dictionaries containing the selected video metadata.

        Parameters:
            video_ids (List[str]): A list of YouTube video ID strings.

        Returns:
            List[dict]: A list of dictionaries with the selected metadata for each video.
        """

        video_info_list = []

        # Iterate over the video ID list in batches of 50 (YouTube API limit)
        for i in range(0, len(video_ids), 50):
            request = self.youtube.videos().list(
                part="snippet,contentDetails,statistics",
                id=','.join(video_ids[i:i + 50])
            )

            # Execute the API request
            response = request.execute()

            # Iterate over each video in the response and extract the desired fields
            for video in response['items']:
                video_info = {
                    'video_id':       video['id'],
                    'channel_title':  video['snippet'].get('channelTitle'),
                    'title':          video['snippet'].get('title'),
                    'description':    video['snippet'].get('description'),
                    'tags':           video['snippet'].get('tags'),
                    'published_at':   video['snippet'].get('publishedAt'),
                    'view_count':     video['statistics'].get('viewCount'),
                    'like_count':     video['statistics'].get('likeCount'),
                    'favorite_count': video['statistics'].get('favoriteCount'),
                    'comment_count':  video['statistics'].get('commentCount'),
                    'duration':       video['contentDetails'].get('duration'),
                    'definition':     video['contentDetails'].get('definition'),
                    'caption':        video['contentDetails'].get('caption')
                }

                video_info_list.append(video_info)

        return video_info_list

    def get_video_details(self, video_ids: List[str]) -> pd.DataFrame:
        """Returns a pandas DataFrame containing details for the provided video IDs.

        Parameters:
            video_ids (List[str]): A list of YouTube video ID strings.

        Returns:
            pd.DataFrame: A DataFrame with one row per video and columns
                          for each metadata field.
        """
        video_info_list = self._fetch_video_details(video_ids)
        df = pd.DataFrame(video_info_list)
        return df

## Final Phase: Generating Video IDs and Saving the DataFrame as a Delta Table

### Generating Video IDs and Returning the pandas DataFrame

In [ ]:
# List of playlists to extract video IDs from
play_complete = [
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3qQB5Pz6ZSTTDLu0BjAJYNf',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3oNcr8es3ov4_4DF8K0Ps6-',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3oeP7PBttxM7esceVXD63_v',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3pXjOUhfPVH3EhW4WMHVYPh',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3rJk6UKP_npByDuE7v1WSdt',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3pQwRz1FORZdChMaNZaR3pu',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3oZGs7Z_sCtp4ND_FLqTssn',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3pdP0S8WTmL5tKgPSZb-rME',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3pKAZxzSqWTfWRyPFHmSS5e',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3pXDttKKp8mlVKDitxsYDAp',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3rYbRYgJbM1ifITbNkqaTsM',
    'https://www.youtube.com/playlist?list=PL1v8zpldgH3pR7LPuidEZK68kS6AaU1y7',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6Hn0vK8co82zjQtt3T2Nkqc',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6Ec-XTbcX1uRg2_u4xOEky0',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6Hn0vK8co82zjQtt3T2Nkqc',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6E7jZ9sN_xHwSHOdjUxUW_b',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6FcbHlDzbVzf3TVgxzxK7lr',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6FIVXnB3nj6razI_m4PKoBC',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6Gqf52H_kkdKTRdn3JMBmDo',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6F6wUI9tvS_Gw1vaFAx6rd6',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6Gl29AoE31iwdVwSG-KnDzF',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6Hmo-Hbqp00dRCrDcOV5AYr',
    'https://www.youtube.com/playlist?list=PLkDaE6sCZn6GMoA0wbpJLi3t34Gd8l0aK'
]

video_id = []

# Loop over each playlist and collect all video IDs
for playlist in play_complete:
    p = Playlist(playlist)
    for url in p.video_urls:
        vid_id = extract.video_id(url)
        video_id.append(vid_id)

# Instantiate the class and fetch video details
youtube_client = youtube_client_obj
youtube_video_details = YouTubeVideoDetails(youtube_client_obj)

df_videos = youtube_video_details.get_video_details(video_id)
print(f"Total videos retrieved: {df_videos.shape[0]}")

### Uploading to S3 as a Parquet File

In [ ]:
# Prompt user for AWS credentials securely
aws_access_key_id = getpass('Enter your AWS Access Key ID: ')
aws_secret_access_key = getpass('Enter your AWS Secret Access Key: ')

# Set credentials as environment variables
os.environ['AWS_ACCESS_KEY_ID'] = aws_access_key_id
os.environ['AWS_SECRET_ACCESS_KEY'] = aws_secret_access_key

# Create the S3 client and save the DataFrame locally as a Parquet file
s3_client = boto3.client('s3')
df_videos.to_parquet('/content/parquet/video_details.parquet')

# Upload the Parquet file(s) to the S3 bucket
for file in os.listdir('/content/parquet'):
    if file.endswith('.parquet'):
        bucket_name = 'youtube-video-details'
        file_path = f'video_data/raw/parquet/{file}'
        s3_client.upload_file(f'/content/parquet/{file}', bucket_name, file_path)
        print(f"Uploaded: {file} -> s3://{bucket_name}/{file_path}")